# Lab 5 - Getting Started with LLM Usage (Hugging Face Transformers)

**Day 2 - HF Transformers** | Estimated time: **40 minutes** | Runs on CPU

So far you have *trained* models. Today you **use** pre-trained models directly, the way most applications do. In this lab you will:

1. Run four tasks in a few lines each with `pipeline()`
2. Look under the hood: **encode** text, run the model, and turn logits into a **prediction**
3. **Generate text** with a small instruction-tuned LLM, and see the chat template it really receives
4. Turn sentences into **embeddings** and compare meaning, not keywords
5. Build a **semantic search** engine over a company knowledge base with **FAISS** (and LangChain)
6. Combine search and generation into a tiny **retrieval-augmented** answer
7. *(Optional)* Store the same vectors in **Pinecone**, a managed cloud vector database

> **How to run:** click inside a code cell and press **Shift+Enter**. Run the cells in order, top to bottom.
> The first run downloads about 1.3 GB of models (skipped if your trainer prefetched them).

In [ ]:
# Setup: quieter logs, UTF-8 output on Windows, and settings from the repository's .env file
import os, sys, time, json, warnings
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
from pathlib import Path
from dotenv import load_dotenv
for folder in [Path.cwd(), *Path.cwd().parents][:4]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

import torch, transformers, sentence_transformers, faiss
torch.manual_seed(42)
print("torch                ", torch.__version__)
print("transformers         ", transformers.__version__)
print("sentence-transformers", sentence_transformers.__version__)
print("faiss                ", faiss.__version__)

# The company knowledge base used in Parts 4-7 (and again in Lab 6)
KB_PATH = Path("../data/helpdesk_kb.json")
kb = json.loads(KB_PATH.read_text(encoding="utf-8"))
print(f"\nKnowledge base: {len(kb)} articles from the (fictional) company Kaveri Systems")

---
## Part 1 - Pipelines: a pre-trained model in three lines (6 min)

`pipeline(task, model=...)` bundles the three steps every model needs:
**tokenize** the text, run the **model**, and **post-process** the raw output into labels or text.

In [ ]:
from transformers import pipeline

sentiment = pipeline("text-classification",
                     model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

reviews = [
    "The new update made the app so much faster. Love it!",
    "Support took three days to reply and didn't fix anything.",
    "Not bad at all.",
]
for review, result in zip(reviews, sentiment(reviews)):
    print(f"{result['label']:<8} {result['score']:.3f}  {review}")

This model was **fine-tuned for one task**, sentiment, with exactly two labels.
What if you need labels nobody trained a model for? **Zero-shot classification** uses a model trained
on *natural language inference* ("does sentence A imply sentence B?") to score **any labels you invent at run time**.

In [ ]:
zero_shot = pipeline("zero-shot-classification",
                     model="MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33")

tickets = [
    "My laptop won't connect to the office VPN since this morning",
    "How many days of annual leave can I carry over to next year?",
    "The hotel in Pune cost more than the limit, will I still get reimbursed?",
    "The projector in the Chamundi boardroom is broken",
]
labels = ["IT support", "HR", "Finance", "Facilities"]

for t in tickets:
    r = zero_shot(t, candidate_labels=labels)
    print(f"{r['labels'][0]:<11} {r['scores'][0]:.2f}  {t}")

> **Try it:** change `labels` to something else (for example `["urgent", "not urgent"]`) and run the cell again.
> No training needed; that's the power, and the risk, of zero-shot models: they'll always pick *something*.

**Your own model from Lab 4.** If you pushed your fine-tuned DistilBERT to the Hub in Lab 4, load it by its repo id.
Set `MY_REPO` to `your-username/distilbert-sst2-lab` and run the cell (it is skipped if left empty).

In [ ]:
MY_REPO = ""   # for example "your-username/distilbert-sst2-lab"

if MY_REPO:
    mine = pipeline("text-classification", model=MY_REPO)   # uses your HF login for private repos
    for review, result in zip(reviews, mine(reviews)):
        print(f"{result['label']:<8} {result['score']:.3f}  {review}")
else:
    print("Skipped: set MY_REPO to try the model you pushed in Lab 4.")

---
## Part 2 - Under the hood: encode, predict, decode (6 min)

Let's do by hand what the sentiment pipeline just did. These are the same three building blocks you used in Day 1:
`AutoTokenizer`, `AutoModelFor...`, and the model's `id2label` mapping.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_ID = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
model.eval()

# 1) ENCODE: text -> token IDs (a PyTorch tensor, padded to the same length)
batch = tokenizer(reviews, padding=True, truncation=True, return_tensors="pt")
print("input_ids shape:", tuple(batch["input_ids"].shape), " (sentences, tokens)")
print("First sentence as tokens:", tokenizer.convert_ids_to_tokens(batch["input_ids"][0])[:14])

In [ ]:
# 2) PREDICT: one forward pass. no_grad() = we're not training, so skip gradient bookkeeping.
with torch.no_grad():
    logits = model(**batch).logits
print("logits:\n", logits)

# 3) DECODE: logits -> probabilities -> label names
probs = torch.softmax(logits, dim=-1)
for review, p in zip(reviews, probs):
    label_id = int(p.argmax())
    print(f"{model.config.id2label[label_id]:<8} {p[label_id]:.3f}  {review}")

The numbers match Part 1 exactly. A pipeline is a convenience wrapper, not magic.

**Logits** are raw, unbounded scores. **Softmax** turns them into probabilities that sum to 1.
Remember this: an LLM generating text does exactly the same thing, but over a vocabulary of ~150,000 tokens, once per generated token.

---
## Part 3 - Text generation with a small LLM (8 min)

**Qwen2.5-0.5B-Instruct** is a 0.5-billion-parameter, instruction-tuned LLM. That's about 1/1000th the size of frontier models,
but small enough to run on this CPU VM. It's a decoder-only transformer, the same architecture family as GPT.

In [ ]:
GEN_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
t0 = time.time()
generator = pipeline("text-generation", model=GEN_MODEL, dtype=torch.float32)
print(f"Loaded {GEN_MODEL} in {time.time() - t0:.0f}s "
      f"({sum(p.numel() for p in generator.model.parameters()) / 1e6:.0f}M parameters)")

messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer in at most three sentences."},
    {"role": "user", "content": "What is a vector database, and why do AI apps use one?"},
]
t0 = time.time()
out = generator(messages, max_new_tokens=120, do_sample=False)
print(f"\n[{time.time() - t0:.1f}s]", out[0]["generated_text"][-1]["content"])

### What the model *actually* sees

Chat models don't receive a list of messages. The **chat template** flattens them into one string with special tokens
that mark each role. Qwen uses the **ChatML** format, one of the prompt formats you'll meet again on Day 4.

In [ ]:
prompt_text = generator.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt_text)
print("Prompt length:", len(generator.tokenizer(prompt_text)["input_ids"]), "tokens")

### Greedy vs sampling

`do_sample=False` (**greedy**) always picks the most likely next token, so the output is the same every time.
With `do_sample=True`, the next token is **sampled** from the probability distribution; `temperature` sharpens (<1) or flattens (>1) it.
You'll explore this much more with a large model in Lab 6.

In [ ]:
prompt = [{"role": "user", "content": "Suggest a name for a coffee shop in Mysuru. Reply with the name only."}]

print("Greedy (3 runs):")
for _ in range(3):
    print("  ", generator(prompt, max_new_tokens=15, do_sample=False)[0]["generated_text"][-1]["content"])

for temp in [0.7, 1.5]:
    print(f"Sampling, temperature={temp} (3 runs):")
    for _ in range(3):
        o = generator(prompt, max_new_tokens=15, do_sample=True, temperature=temp, top_p=0.95)
        print("  ", o[0]["generated_text"][-1]["content"])

### The limitation that matters most

Ask the model about something it **can't** know: an internal policy of a company that doesn't exist publicly.

In [ ]:
QUESTION = "At Kaveri Systems, how many unused annual leave days can I carry over to next year?"

closed_book = generator([{"role": "user", "content": QUESTION}], max_new_tokens=80, do_sample=False)
print(closed_book[0]["generated_text"][-1]["content"])

Read the answer closely. The model may start by admitting it has no information, and then **guess anyway** with a plausible-sounding
"typical" number. Either way, it can't give the real answer: it has never seen Kaveri Systems' policies.
A confident, fluent, unsupported answer is a **hallucination**. Keep this answer in mind; in Part 6 you'll fix it by retrieving the right policy first.

---
## Part 4 - Embeddings: meaning as numbers (6 min)

An **embedding model** turns a whole sentence into one vector (here, 384 numbers).
Sentences with similar **meaning** get vectors that point in similar directions, even when they share no words.
We use `all-MiniLM-L6-v2` from the `sentence-transformers` library (22M parameters, fast on CPU).

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = [
    "How do I reset my password?",
    "I forgot my login credentials",
    "What is the hotel limit for business trips?",
    "How much can I spend on accommodation when travelling?",
    "The cafeteria serves free filter coffee",
]
vectors = embedder.encode(sentences, normalize_embeddings=True)
print("Embedding matrix shape:", vectors.shape, " (sentences, dimensions)")
print("First 8 numbers of sentence 0:", vectors[0][:8].round(3))

In [ ]:
import pandas as pd

sim = embedder.similarity(vectors, vectors).numpy()     # cosine similarity, -1..1
short = [s[:28] + "..." for s in sentences]
pd.DataFrame(sim, index=short, columns=range(len(sentences))).round(2)

Sentences 0 and 1 share **no keywords** ("reset password" vs "forgot login credentials") yet score high.
Sentences 2 and 3 are the same: "hotel" vs "accommodation". That's what makes semantic search better than keyword search.

---
## Part 5 - Semantic search with a vector store (FAISS) (8 min)

A **vector store** keeps many embeddings and quickly finds the ones nearest to a query vector.
**FAISS** (Facebook AI Similarity Search) is a fast open-source library that runs locally, with no server and no account.

We index the 36 articles of the Kaveri Systems helpdesk knowledge base.

In [ ]:
import numpy as np

docs = [f"{a['title']}. {a['text']}" for a in kb]
t0 = time.time()
doc_vectors = embedder.encode(docs, normalize_embeddings=True, batch_size=16)
print(f"Embedded {len(docs)} articles in {time.time() - t0:.1f}s -> {doc_vectors.shape}")

# IndexFlatIP = exact search by inner product. With normalised vectors, inner product = cosine similarity.
index = faiss.IndexFlatIP(doc_vectors.shape[1])
index.add(doc_vectors.astype(np.float32))
print("Vectors in the index:", index.ntotal)

In [ ]:
def search(query, k=3):
    q = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q, k)
    return [(float(s), kb[i]) for s, i in zip(scores[0], ids[0])]

queries = [
    "I can't remember my login details",
    "my internet at home is slow, will the company pay for it?",
    "I clicked a weird link in an email",
    "can I get a bigger laptop for development work?",
]
for q in queries:
    print(f"\nQ: {q}")
    for score, art in search(q):
        print(f"   {score:.2f}  {art['id']}  {art['title']}")

Compare with plain **keyword** search on the first query. The password article never uses the words "remember", "login" or "details".

In [ ]:
def keyword_search(query, k=3):
    words = {w.strip("?,.!").lower() for w in query.split() if len(w) > 3}
    scored = [(sum(w in (a["title"] + " " + a["text"]).lower() for w in words), a) for a in kb]
    return sorted(scored, key=lambda x: -x[0])[:k]

for hits, art in keyword_search(queries[0]):
    print(f"   {hits} keyword hits  {art['id']}  {art['title']}")

Keyword matching rewards common words; semantic search matches **intent**. In production, many systems combine both (**hybrid search**).

### The same thing with LangChain

**LangChain** wraps embedding models and vector stores behind standard interfaces, so you can swap FAISS for Pinecone,
Azure AI Search or pgvector without rewriting your app. Lab 7 uses LangChain again, for agents.

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

lc_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                      encode_kwargs={"normalize_embeddings": True})
lc_docs = [Document(page_content=d, metadata={"id": a["id"], "category": a["category"]})
           for d, a in zip(docs, kb)]
store = FAISS.from_documents(lc_docs, lc_embeddings)

# Metadata filtering: only search Finance articles
for doc, score in store.similarity_search_with_score("how much for a hotel night?", k=2,
                                                     filter={"category": "Finance"}):
    print(f"{score:.3f}  {doc.metadata['id']}  {doc.page_content[:70]}...")

store.save_local("outputs/faiss_kb")          # persist the index to disk...
reloaded = FAISS.load_local("outputs/faiss_kb", lc_embeddings,
                            allow_dangerous_deserialization=True)   # ...and load it back (only load files you created!)
print("Reloaded index holds", reloaded.index.ntotal, "vectors")

> LangChain's FAISS reports a **distance** (lower = closer), while our raw FAISS index reported a **similarity** (higher = closer).
> Always check which one a vector store returns before you set a threshold.

---
## Part 6 - Retrieval + generation = grounded answers (5 min)

Now fix the hallucination from Part 3. **Retrieval-augmented generation (RAG)**:
1. **Retrieve** the most relevant articles for the question
2. Put them in the prompt as **context**
3. Instruct the model to answer **only from that context**

In [ ]:
RAG_SYSTEM = ("You answer employee questions using ONLY the context below. "
              "Answer in one or two sentences and end with the article id in brackets. "
              "If the context does not contain the answer, reply exactly: I don't know.")

def rag_answer(question, k=2, system=RAG_SYSTEM, show_context=True):
    hits = search(question, k=k)
    context = "\n".join(f"[{a['id']}] {a['title']}: {a['text']}" for _, a in hits)
    if show_context:
        print("Retrieved:", ", ".join(a["id"] for _, a in hits))
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    return generator(msgs, max_new_tokens=100, do_sample=False)[0]["generated_text"][-1]["content"]

print("CLOSED BOOK (Part 3):", closed_book[0]["generated_text"][-1]["content"][:200], "...\n")
print("WITH RETRIEVAL:", rag_answer(QUESTION))

In [ ]:
for q in ["What is the hotel limit per night in Bengaluru?",
          "Who is the CEO of Kaveri Systems?"]:          # not in the knowledge base!
    print(f"\nQ: {q}")
    print("A:", rag_answer(q))

Look carefully at both answers:

* Retrieval supplied the **facts**; the small model only had to **read** them. But notice it assumed Bengaluru is a "metro city".
  The article never says which cities count as metros, so even a grounded answer can rest on an unstated assumption.
* The knowledge base has nothing about the CEO, so the right answer is "I don't know". The retriever still returned the two
  *least bad* articles (vector search always returns *something*), and the 0.5B model typically **invents a name** despite the instruction.
  Larger models follow instructions far better, which is what you'll test in **Lab 6**.

Small models are also very **sensitive to prompt wording**. The next cell asks the same two questions with a system
prompt that is only slightly different: it gives the assistant a role and shows an example citation.

In [ ]:
ALT_SYSTEM = ("You are the Kaveri Systems helpdesk assistant. Answer using ONLY the context. "
              "Answer in one or two sentences and cite the article id, like [KB-013]. "
              "If the context does not contain the answer, reply exactly: I don't know.")

for q in ["Who is the CEO of Kaveri Systems?", QUESTION]:
    print(f"\nQ: {q}")
    print("A:", rag_answer(q, system=ALT_SYSTEM, show_context=False))

With this wording the model usually says "I don't know" about the CEO. Now look at the **citations**: both prompts asked
for the article id, and the 0.5B model rarely includes it, whichever wording you use. A small change can fix one behaviour
and leave another broken, so every prompt change needs re-testing on *several* questions. That's the
iterative prompting loop you'll practise on Day 3.

> **Try it:** ask your own question about the knowledge base (`rag_answer("...")`), then one it can't answer.

---
## Part 7 (optional) - A managed vector database: Pinecone

FAISS lives inside your Python process. **Pinecone** is a hosted vector database: the index lives in the cloud,
survives restarts, scales to billions of vectors, and many apps can share it.

**Only if you want to try it:**

1. Sign up for the free **Starter** plan at https://app.pinecone.io (no credit card).
2. In the Pinecone console open **API Keys** and copy the default key.
3. Add a line `PINECONE_API_KEY=...` to the `.env` file at the repository root, save it, and **restart the kernel**
   (then re-run the Setup cell and the Part 4 + Part 5 cells that create `embedder`, `docs` and `doc_vectors`).

The free plan creates serverless indexes in AWS `us-east-1`. The cells below skip themselves if no key is set.

In [ ]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "")
INDEX_NAME = "kaveri-kb"

if not PINECONE_API_KEY or PINECONE_API_KEY.startswith("<"):
    print("Skipped: no PINECONE_API_KEY in .env (this part is optional).")
else:
    from pinecone import Pinecone, ServerlessSpec
    pc = Pinecone(api_key=PINECONE_API_KEY)
    if not pc.has_index(INDEX_NAME):
        pc.create_index(name=INDEX_NAME, dimension=doc_vectors.shape[1], metric="cosine",
                        spec=ServerlessSpec(cloud="aws", region="us-east-1"))
        print("Created index", INDEX_NAME)
    pindex = pc.index(name=INDEX_NAME)

    # Upsert = insert or update. Each record: id, vector, and metadata you can filter on later.
    records = [{"id": a["id"], "values": v.tolist(),
                "metadata": {"title": a["title"], "category": a["category"], "text": a["text"]}}
               for a, v in zip(kb, doc_vectors)]
    pindex.upsert(vectors=records, namespace="helpdesk", show_progress=False)
    print("Upserted", len(records), "vectors")

In [ ]:
if PINECONE_API_KEY and not PINECONE_API_KEY.startswith("<"):
    time.sleep(5)   # serverless indexes are eventually consistent: give new vectors a moment
    print(pindex.describe_index_stats())
    q = embedder.encode(["can I get a bigger laptop for development work?"], normalize_embeddings=True)[0]
    res = pindex.query(vector=q.tolist(), top_k=3, include_metadata=True, namespace="helpdesk",
                       filter={"category": {"$eq": "IT"}})
    for m in res.matches:
        print(f"{m.score:.2f}  {m.id}  {m.metadata['title']}")

In [ ]:
# Clean up: delete the index when you're done (the free plan allows only a few indexes)
if PINECONE_API_KEY and not PINECONE_API_KEY.startswith("<"):
    pc.delete_index(INDEX_NAME)
    print("Deleted", INDEX_NAME)

The scores match your local FAISS results, because the **vectors** are identical. Only the **storage and serving** changed.

| | FAISS | Pinecone |
|---|---|---|
| Runs | In your process | Managed cloud service |
| Persistence | Save/load files yourself | Automatic |
| Scale | Millions (one machine's RAM) | Billions |
| Metadata filtering | Limited (LangChain adds it) | Built in |
| Cost | Free | Free tier, then usage-based |

---
## Summary

| You did | Why it matters |
|---|---|
| `pipeline()` for classification and zero-shot | Pre-trained models solve many tasks with no training |
| Encode → logits → softmax by hand | Every model, LLMs included, ends in a probability distribution |
| Generated text and saw the chat template | Chat "messages" are just a formatted prompt |
| Embeddings + FAISS semantic search | Retrieval by meaning is the backbone of RAG and agent memory |
| Retrieval + generation | Grounding reduces hallucination, but model quality still matters |

**Next:** Lab 6 uses a large hosted LLM (Azure OpenAI) to explore prompt variations, grounding and responsible AI.